In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
DEL_data=pd.read_csv("../data/raw/delhi.csv")

In [3]:
BLRU_data = pd.read_csv("../data/raw/bangalore.csv")  # Make sure the file isn't named blru.csv or bangalore.csv
PILANI_data = pd.read_csv("../data/raw/pilani.csv")
MUM_data = pd.read_csv("../data/raw/mumbai.csv")
NAG_data = pd.read_csv("../data/raw/nagpur.csv")

In [4]:
BLRU_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5476 entries, 0 to 5475
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   date                           5476 non-null   str    
 1   time_gmt_5_30                  5476 non-null   str    
 2   latitude                       5476 non-null   float64
 3   longitude                      5476 non-null   float64
 4   city_id                        5476 non-null   int64  
 5   city_name                      5476 non-null   str    
 6   weather_code                   5476 non-null   int64  
 7   temp_mean_c                    5476 non-null   float64
 8   temp_max_c                     5476 non-null   float64
 9   temp_min_c                     5476 non-null   float64
 10  wind_speed_max_kmh             5476 non-null   float64
 11  wind_gusts_max_kmh             5476 non-null   float64
 12  wind_direction_dominant_deg    5476 non-null   float64
 13 

In [5]:
import pandas as pd

def preprocess_weather_data(df):
    # Convert date to datetime format
    df['date'] = pd.to_datetime(df['date'])
    
    # Extract day, month, year, and day of week
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['day_of_week'] = df['date'].dt.dayofweek
    
    # Convert city_id and weather_code to categorical
    df['city_id'] = df['city_id'].astype('category')
    df['weather_code'] = df['weather_code'].astype('category')
    
    # Drop date, time, time_gmt_5_30, city_name, and cloud_cover max/min
    cols_to_drop = [
        'date', 
        'time',               # <-- Added 'time' to the drop list
        'time_gmt_5_30', 
        'city_name', 
        'cloud_cover_max_pct', 
        'cloud_cover_min_pct'
    ]
    df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    return df

# Create a list of your dataframes
dataframes = [DEL_data, BLRU_data, PILANI_data, MUM_data, NAG_data]

# Apply the function to each dataframe
DEL_data, BLRU_data, PILANI_data, MUM_data, NAG_data = [preprocess_weather_data(df.copy()) for df in dataframes]

# Verify the changes on one of them
DEL_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5476 entries, 0 to 5475
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   latitude                       5476 non-null   float64 
 1   longitude                      5476 non-null   float64 
 2   city_id                        5476 non-null   category
 3   weather_code                   5476 non-null   category
 4   temp_mean_c                    5476 non-null   float64 
 5   temp_max_c                     5476 non-null   float64 
 6   temp_min_c                     5476 non-null   float64 
 7   wind_speed_max_kmh             5476 non-null   float64 
 8   wind_gusts_max_kmh             5476 non-null   float64 
 9   wind_direction_dominant_deg    5476 non-null   float64 
 10  shortwave_radiation_sum_mj_m2  5476 non-null   float64 
 11  daylight_duration_s            5476 non-null   float64 
 12  precipitation_sum_mm           5476 non-null 

In [9]:
import numpy as np

def apply_cyclic_transformations(df):
    df = df.copy()
    
    # 1. Convert Wind Direction to Cyclic (Radians)
    df['wind_dir_sin'] = np.sin(np.radians(df['wind_direction_dominant_deg']))
    df['wind_dir_cos'] = np.cos(np.radians(df['wind_direction_dominant_deg']))
    
    # 2. Convert Month to Cyclic (1 to 12)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    # 3. Convert Day to Cyclic (Assuming 31 days max for the cycle)
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
    
    # 4. Drop the original raw integer/degree columns to prevent collinearity
    cols_to_drop = [
        'wind_direction_dominant_deg', 
        'month', 
        'day'
    ]
    
    return df.drop(columns=cols_to_drop, errors='ignore')



In [10]:
DEL_data=apply_cyclic_transformations(DEL_data)

In [17]:

BLRU_data=apply_cyclic_transformations(BLRU_data)


KeyError: 'wind_direction_dominant_deg'

In [13]:
PIL_data=apply_cyclic_transformations(PILANI_data)

In [14]:
MUMBAI_data=apply_cyclic_transformations(MUM_data)

In [15]:
NAGPUR_data=apply_cyclic_transformations(NAG_data)

In [19]:
def time_series_split(df, train_ratio=0.70, val_ratio=0.15):
    """
    Splits a dataframe chronologically into Train, Validation, and Test sets.
    Assumes the dataframe is already sorted by time.
    """
    n = len(df)
    
    # Calculate the integer indices for the splits
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    
    # Slice the dataframe using .iloc (and use .copy() to avoid SettingWithCopy warnings later)
    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()
    
    return train_df, val_df, test_df

# Apply the split to your datasets (Example for Delhi)
train_DEL, val_DEL, test_DEL = time_series_split(DEL_data)

# You can do the same for the others
train_BLRU, val_BLRU, test_BLRU = time_series_split(BLRU_data)
train_PIL, val_PIL, test_PIL = time_series_split(PIL_data)
train_MUM, val_MUM, test_MUM = time_series_split(MUMBAI_data)
train_NAG, val_NAG, test_NAG = time_series_split(NAGPUR_data)

# Verify the split sizes using Delhi as an example
print(f"Total shape: {DEL_data.shape}")
print(f"Train shape: {train_DEL.shape} - {len(train_DEL)/len(DEL_data):.0%}")
print(f"Val shape:   {val_DEL.shape} - {len(val_DEL)/len(DEL_data):.0%}")
print(f"Test shape:  {test_DEL.shape} - {len(test_DEL)/len(DEL_data):.0%}")

Total shape: (5476, 30)
Train shape: (3833, 30) - 70%
Val shape:   (821, 30) - 15%
Test shape:  (822, 30) - 15%


In [20]:
import pandas as pd

# 1. Join all Training sets
train_data = pd.concat(
    [train_DEL, train_BLRU, train_PIL, train_MUM, train_NAG], 
    axis=0, 
    ignore_index=True
)

# 2. Join all Validation sets
val_data = pd.concat(
    [val_DEL, val_BLRU, val_PIL, val_MUM, val_NAG], 
    axis=0, 
    ignore_index=True
)

# 3. Join all Test sets
test_data = pd.concat(
    [test_DEL, test_BLRU, test_PIL, test_MUM, test_NAG], 
    axis=0, 
    ignore_index=True
)

# Verify the combined shapes
print(f"Combined Train shape: {train_data.shape}")
print(f"Combined Val shape:   {val_data.shape}")
print(f"Combined Test shape:  {test_data.shape}")

Combined Train shape: (19165, 30)
Combined Val shape:   (4105, 30)
Combined Test shape:  (4110, 30)


In [21]:
import os
import shutil

processed_dir = "../data/processed"

# 1. Clear the processed directory completely
if os.path.exists(processed_dir):
    shutil.rmtree(processed_dir) # Deletes the folder and everything inside it

# 2. Recreate the empty directory
os.makedirs(processed_dir)

# 3. Save the new combined dataframes as CSVs
# Setting index=False ensures the arbitrary row numbers aren't saved as a new column
train_data.to_csv(f"{processed_dir}/train.csv", index=False)
val_data.to_csv(f"{processed_dir}/val.csv", index=False)
test_data.to_csv(f"{processed_dir}/test.csv", index=False)

print(f"Directory '{processed_dir}' cleared.")
print("Successfully saved: train.csv, val.csv, and test.csv")

Directory '../data/processed' cleared.
Successfully saved: train.csv, val.csv, and test.csv


In [22]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 19165 entries, 0 to 19164
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   latitude                       19165 non-null  float64 
 1   longitude                      19165 non-null  float64 
 2   city_id                        19165 non-null  int64   
 3   weather_code                   19165 non-null  category
 4   temp_mean_c                    19165 non-null  float64 
 5   temp_max_c                     19165 non-null  float64 
 6   temp_min_c                     19165 non-null  float64 
 7   wind_speed_max_kmh             19165 non-null  float64 
 8   wind_gusts_max_kmh             19165 non-null  float64 
 9   shortwave_radiation_sum_mj_m2  19165 non-null  float64 
 10  daylight_duration_s            19165 non-null  float64 
 11  precipitation_sum_mm           19165 non-null  float64 
 12  cloud_cover_mean_pct           19165 non-nu